# Lab 3.4 — Hydrogen's spectral lines

**Goal:** compute the wavelengths of light hydrogen emits when its electron jumps between energy levels, and compare them with measured values.

In the Bohr model the energy of level *n* is

$$E_n = -\frac{13.6\ \text{eV}}{n^2}$$

When the electron drops from level $n_u$ to $n_l$, it emits one photon with energy $\Delta E = E_{n_u} - E_{n_l}$ and wavelength $\lambda = hc / \Delta E$.

In [ ]:
import sys; sys.path.append("..")
from constants import E_R, hc_eV_nm
import numpy as np
import matplotlib.pyplot as plt
import csv

## 1. Energy levels

Complete `energy_level(n)`, then run the cell to print levels 1–6.

In [ ]:
def energy_level(n):
    # Energy of level n in eV (negative = bound)
    return -E_R / n**2

for n in range(1, 7):
    print(f"n = {n}:  E = {energy_level(n):7.3f} eV")

## 2. Every transition between n = 1 and n = 6

Which ones are visible (380–750 nm)?

In [ ]:
def transition_wavelength_nm(n_upper, n_lower):
    dE = energy_level(n_upper) - energy_level(n_lower)   # eV, positive
    return hc_eV_nm / dE

SERIES = {1: "Lyman (UV)", 2: "Balmer", 3: "Paschen (IR)", 4: "Brackett (IR)", 5: "Pfund (IR)"}
visible = []
for n_lower in range(1, 6):
    for n_upper in range(n_lower + 1, 7):
        lam = transition_wavelength_nm(n_upper, n_lower)
        tag = "VISIBLE" if 380 <= lam <= 750 else ""
        if tag:
            visible.append((n_upper, n_lower, lam))
        print(f"{n_upper} -> {n_lower}  {SERIES[n_lower]:14s} {lam:9.1f} nm  {tag}")

print("\nVisible lines:", [f"{u}->{l}" for u, l, _ in visible])

## 3. Compare with measurement

`data/hydrogen_lines.csv` holds measured Balmer wavelengths (in air). How close is the Bohr model?

In [ ]:
with open("../../data/hydrogen_lines.csv") as f:
    rows = list(csv.DictReader(f))

print(f"{'line':10s} {'measured':>9s} {'Bohr':>9s} {'error':>7s}")
for r in rows:
    measured = float(r["wavelength_nm_air"])
    bohr = transition_wavelength_nm(int(r["n_upper"]), int(r["n_lower"]))
    print(f"{r['name']:10s} {measured:9.2f} {bohr:9.2f} {100*(bohr-measured)/measured:6.2f}%")

The Bohr values are within about 0.1%. (Most of the gap: the measured values are in air, and the model ignores the proton's motion.)

## 4. Draw the visible spectrum

In [ ]:
def wavelength_to_rgb(lam):
    # Approximate RGB colour for a visible wavelength in nm
    if   380 <= lam < 440: r, g, b = (440 - lam) / 60, 0, 1
    elif 440 <= lam < 490: r, g, b = 0, (lam - 440) / 50, 1
    elif 490 <= lam < 510: r, g, b = 0, 1, (510 - lam) / 20
    elif 510 <= lam < 580: r, g, b = (lam - 510) / 70, 1, 0
    elif 580 <= lam < 645: r, g, b = 1, (645 - lam) / 65, 0
    elif 645 <= lam <= 750: r, g, b = 1, 0, 0
    else: r, g, b = 0, 0, 0
    return (r, g, b)

fig, ax = plt.subplots(figsize=(10, 2.2))
ax.set_facecolor("black")
for u, l, lam in visible:
    ax.axvline(lam, color=wavelength_to_rgb(lam), lw=4)
    ax.text(lam, 1.03, f"{u}→{l}\n{lam:.0f} nm", ha="center", va="bottom", fontsize=9, transform=ax.get_xaxis_transform())
ax.set_xlim(380, 750)
ax.set_yticks([])
ax.set_xlabel("Wavelength (nm)")
ax.set_title("Hydrogen's visible (Balmer) emission lines", pad=34)
plt.tight_layout()
plt.show()

## 5. Newton vs quantum: the gaps close

Classically, an electron could have **any** energy. Plot the gap between neighbouring levels, $E_{n+1} - E_n$, as $n$ grows. What happens to the gap, and what does that tell you about where the quantum and classical pictures meet?

In [ ]:
n = np.arange(1, 31)
gap = energy_level(n + 1) - energy_level(n)

plt.figure(figsize=(7, 4))
plt.semilogy(n, gap, "o-")
plt.xlabel("n")
plt.ylabel("Gap to next level (eV, log scale)")
plt.title("Energy gaps shrink as n grows (correspondence principle)")
plt.grid(alpha=0.3)
plt.show()
print(f"gap 1→2: {gap[0]:.2f} eV;  gap 30→31: {gap[-1]:.5f} eV")

## Exercises

1. **[Core]** Which Balmer line has the shortest visible wavelength in your list? Which transition makes it?
2. **[Extension]** The Lyman series (ending at n = 1) is all ultraviolet. Show this with one line of reasoning, then check with code.
3. **[Challenge]** Repeat the comparison using the *reduced mass* $\mu = m_e m_p / (m_e + m_p)$ (multiply $E_R$ by $\mu/m_e$). Does the error shrink?